In [62]:
import folium
from folium.plugins import MarkerCluster
import numpy as np
import webbrowser
import pandas as pd
import branca.colormap as cm

In [63]:
# Načítanie dát
edges = pd.read_csv("../data/edges.csv").rename(columns=lambda x: x.strip())[["# source", "target", "distance", "airline"]].drop_duplicates()
nodes = pd.read_csv("../data/nodes.csv").rename(columns=lambda x: x.strip())[["name","city","country","latitude","longitude","altitude"]]
nodes = pd.merge(nodes, pd.read_csv("../data/continents.csv"),on="country")

# Vyčistíme edges od letov, ktoré nemajú súradnice v nodes
edges = edges[edges["# source"].isin(nodes.index) & edges["target"].isin(nodes.index)]

In [64]:
class Map:
    def __init__(self, center=[20, 0], zoom_start=2, tiles="cartodb positron"):
        self.m = folium.Map(
            location=center, zoom_start=zoom_start, tiles=tiles, prefer_canvas=True
        )

    def add_nodes(self, df, show_text=False):
        continents_list = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']
        for idx, r in df.iterrows():
            # Červený bod
            folium.CircleMarker(
                [r["latitude"], r["longitude"]],
                radius=6, color="white", weight=2, fill=True,
                fill_color="#DD0A0A", fill_opacity=1.0, zorder=100
            ).add_to(self.m)

            if show_text:
                raw_name = r.get("city", r.get("name", str(idx)))
                name = raw_name.replace("International Airport", "").replace("Airport", "").strip().rstrip(',')
                if raw_name in continents_list:
                    bg_color = "#115E07" # Veľmi svetlá zelená (Mint)
                else:
                    bg_color = "white"
                traffic_val = r.get("gateway_traffic")
                if traffic_val is not None:
                    badge_color = "#1A237E"
                else:
                    traffic_val = r.get("in_traffic")
                    badge_color = "#115E07"
                    
                    if traffic_val is None:
                        traffic_val = r.get("traffic")
                        badge_color = "#546E7A"
                place = r.get('place', 'up')
                
                # HTML pre vnútro karty
                inner_html = f"""
                    <div style="background: {bg_color}; color: {"black" if bg_color == "white" else "white"}; border: 2px solid {badge_color}; border-radius: 8px; 
                                width: 200px; box-shadow: 0 4px 10px rgba(0,0,0,0.2); overflow: hidden; 
                                display: flex; flex-direction: column; font-family: Arial;">
                        <div style="padding: 5px; text-align: center; font-size: 12pt; font-weight: 900; text-transform: uppercase;">
                            {name}
                        </div>
                        <div style="background: {badge_color}; color: white; font-size: 12pt; font-weight: bold; 
                                    padding: 2px 0; text-align: center;">
                            TRAFFIC: {int(traffic_val)}
                        </div>
                    </div>
                """

                # Logika smerovania trojuholníka (CSS triangle)
                tri_base = "8px solid transparent"
                tri_color = f"8px solid {badge_color if place == 'up' else 'white'}"
                
                if place == 'up':
                    flex_dir, tri_style = "column", f"border-top: {tri_color}; border-left: {tri_base}; border-right: {tri_base};"
                elif place == 'bottom':
                    flex_dir, tri_style = "column-reverse", f"border-bottom: 8px solid white; border-left: {tri_base}; border-right: {tri_base};"
                elif place == 'left':
                    flex_dir, tri_style = "row", f"border-left: 8px solid {badge_color}; border-top: {tri_base}; border-bottom: {tri_base};"
                else: # right
                    flex_dir, tri_style = "row-reverse", f"border-right: 8px solid white; border-top: {tri_base}; border-bottom: {tri_base};"

                html_content = f"""
                    <div style="display: flex; flex-direction: {flex_dir}; align-items: center; justify-content: center;">
                        {inner_html}
                        <div style="width: 0; height: 0; {tri_style}"></div>
                    </div>
                """

                folium.Marker(
                    [r["latitude"], r["longitude"]],
                    icon=folium.DivIcon(
                        icon_size=(160, 120),
                        icon_anchor=r['anchor'],
                        html=html_content
                    )
                ).add_to(self.m)

    def add_edges(self, flow_df, scaling=5.0, offset_dist=1.5, value_show=False):
        log_f = np.log1p(flow_df["flight_count"])
        colormap = cm.LinearColormap(
            colors=["#CF6464", "#00095D"],
            vmin=log_f.min(),
            vmax=log_f.max(),
        )

        for _, row in flow_df.iterrows():
            try:
                # 1. Základné body
                s_lat, s_lon = map(float, str(row["start"]).split(","))
                e_lat, e_lon = map(float, str(row["end"]).split(","))

                # 2. VIRTUÁLNY LON (aby sme mali "rovnú" čiaru cez antimeridián)
                v_e_lon = e_lon
                if e_lon - s_lon > 180:
                    v_e_lon -= 360
                elif e_lon - s_lon < -180:
                    v_e_lon += 360

                # 3. OFFSET (posun) vo virtuálnom priestore
                d_lat = -e_lat + s_lat
                dv_lon = -v_e_lon + s_lon
                dist = np.sqrt(d_lat**2 + dv_lon**2)
                if dist == 0:
                    continue

                perp_lat = (dv_lon / dist) * offset_dist
                perp_lon = -(d_lat / dist) * offset_dist

                # Posunuté virtuálne body
                vs = (s_lat + perp_lat, s_lon + perp_lon)
                ve = (e_lat + perp_lat, v_e_lon + perp_lon)

                # 4. ROZDELENIE NA SEGMENTY (Antimeridián fix)
                def wrap(lon):
                    return (lon + 180) % 360 - 180

                segments = []
                # Kontrola, či virtuálna čiara pretína hranicu 180/-180
                # t.j. či jeden bod je > 180 alebo < -180 po posune
                if (
                    (vs[1] > 180 and ve[1] <= 180)
                    or (vs[1] <= 180 and ve[1] > 180)
                    or (vs[1] < -180 and ve[1] >= -180)
                    or (vs[1] >= -180 and ve[1] < -180)
                ):

                    boundary = 180 if (vs[1] > 180 or ve[1] > 180) else -180
                    t = (boundary - vs[1]) / (ve[1] - vs[1])
                    mid_lat = vs[0] + t * (ve[0] - vs[0])

                    # Rozdelíme na dva segmenty a každý "upraceme" (wrap)
                    segments.append([(vs[0], wrap(vs[1])), (mid_lat, boundary)])
                    segments.append([(mid_lat, -boundary), (ve[0], wrap(ve[1]))])
                else:
                    # Žiadne kríženie, len upraceme štart a cieľ
                    segments.append([(vs[0], wrap(vs[1])), (ve[0], wrap(ve[1]))])

                # 5. KRESLENIE
                log_v = np.log1p(row["flight_count"])
                color = colormap(log_v)
                angle = np.degrees(np.arctan2(d_lat, dv_lon))
                flight_val = int(row["flight_count"])
                if angle > 90 or angle < -90:
                    display_angle = angle - 180
                    # Šípka vľavo (←), zväčšená cez transform a line-height
                    arrow_html = "<span style='font-size: 30pt; line-height: 0; margin-left: 8px; vertical-align: middle;'>→</span>"
                    label_html = f"{flight_val}{arrow_html}"
                    
                else:
                    display_angle = angle
                    # Šípka vpravo (→), zväčšená cez transform a line-height
                    arrow_html = "<span style='font-size: 30pt; line-height: 0; margin-right: 8px; vertical-align: middle;'>←</span>"
                    label_html = f"{arrow_html}{flight_val}"

                for seg in segments:
                    # Kreslíme čiaru
                    folium.PolyLine(
                        seg, color=color, weight=log_v * scaling, opacity=0.7
                    ).add_to(self.m)
                    if value_show:
                        # Text dáme do stredu segmentu, ale len ak je segment dostatočne dlhý
                        # (aby sme nemali text dvakrát na okrajoch mapy pri rozdelení)
                        m_lat = (seg[0][0] + seg[1][0]) / 2
                        m_lon = (seg[0][1] + seg[1][1]) / 2

                        folium.Marker(
                            [m_lat, m_lon],
                            icon=folium.DivIcon(
                                html=f"""<div style="
                                display: flex; 
                                align-items: center;  
                                transform: rotate({-display_angle}deg); 
                                font-size: 17pt; 
                                color: {color}; 
                                font-weight: bold; 
                                white-space: nowrap;">
                                {label_html}
                            </div>
                        """),
                        ).add_to(self.m)

            except Exception as e:
                print(f"Chyba: {e}")

    def showMap(self, filename="mapa_final.html"):
        self.m.save(filename)
        webbrowser.open(filename)

In [65]:
def make_start_end(edges_df):
    new_edges = edges_df.copy()
    new_edges[["start","end"]] = new_edges[["# source", "target"]].apply(lambda row: 
                                           f"{nodes.loc[row['# source'],'latitude']}," +
                                           f"{nodes.loc[row['# source'],'longitude']};"+
                                           f"{nodes.loc[row['target'],'latitude']},"+
                                           f"{nodes.loc[row['target'],'longitude']}", axis=1).str.split(";",expand=True)
    return new_edges

In [66]:
# 1. Musíme zabezpečiť, aby indexy boli INTEGER pre správne párovanie
nodes.index = nodes.index.astype(int)
edges["# source"] = edges["# source"].astype(int)
edges["target"] = edges["target"].astype(int)

# 2. Očista krajín (odstránenie medzier)
nodes["country"] = nodes["country"].astype(str).str.strip()
nodes["continent"] = nodes["continent"].astype(str).str.strip()

# 3. Vyčistíme edges od letov, ktoré nemajú súradnice v nodes
edges = edges[edges["# source"].isin(nodes.index) & edges["target"].isin(nodes.index)].copy()

# 4. PRIDANIE KRAJÍN DO EDGES (toto ti chýbalo)
edges["src_country"] = edges["# source"].map(nodes["country"])
edges["dest_country"] = edges["target"].map(nodes["country"])
edges["src_continent"] = edges["# source"].map(nodes["continent"])
edges["dest_continent"] = edges["target"].map(nodes["continent"])

print(f"Počet hrán po vyčistení: {len(edges)}")
print(f"Ukážka priradených krajín:\n{edges[['# source', 'src_country', 'target', 'dest_country']].head()}")

Počet hrán po vyčistení: 66771
Ukážka priradených krajín:
   # source       src_country  target      dest_country
0         0  Papua New Guinea       2  Papua New Guinea
1         0  Papua New Guinea       3  Papua New Guinea
2         0  Papua New Guinea       1  Papua New Guinea
3         0  Papua New Guinea       4  Papua New Guinea
4         0  Papua New Guinea       4  Papua New Guinea


In [67]:
# 1. Výpočet popularity (trafficu) letísk
traffic = pd.concat([edges["# source"], edges["target"]]).value_counts()
nodes["traffic"] = nodes.index.map(traffic).fillna(0)

# 2. Výber najpoužívanejšieho letiska (Hubu) pre každú krajinu
# Zoradíme podľa trafficu a zahodíme duplikáty krajín
country_hubs = (
    nodes.sort_values(["country", "traffic"], ascending=[True, False])
    .drop_duplicates("country")
    .copy()
)


# !!! TOTO JE TÁ OPRAVA: Nastavíme stĺpec 'country' ako index a očistíme ho
country_hubs["country"] = country_hubs["country"].astype(str).str.strip()
country_hubs.set_index("country", inplace=True)

country_hubs = country_hubs[["name", "latitude", "longitude", "traffic"]].rename(
    columns={"name": "hub_name"}
)

# 3. Agregácia tokov medzi krajinami
flow_df = (
    edges.groupby(["src_country", "dest_country"])
    .size()
    .reset_index(name="flight_count")
)


# 4. Pridanie súradníc (bezpečnejšia verzia)
def get_hub_coords_fixed(c_name, df):
    c_name = str(c_name).strip()
    if c_name in df.index:
        row = df.loc[c_name]
        return f"{row['latitude']},{row['longitude']}"
    return None


flow_df["start"] = flow_df["src_country"].apply(get_hub_coords_fixed, df=country_hubs)
flow_df["end"] = flow_df["dest_country"].apply(get_hub_coords_fixed, df=country_hubs)

# 5. Finálny filter na medzinárodné lety
flow_to_draw = flow_df[flow_df["src_country"] != flow_df["dest_country"]].dropna(
    subset=["start", "end"]
)

print(f"DEBUG: Počet riadkov v country_hubs: {len(country_hubs)}")
print(f"DEBUG: Prvé 3 indexy v country_hubs: {country_hubs.index[:3].tolist()}")
print(f"POČET MEDZINÁRODNÝCH TOKOV NA VYKRESLENIE: {len(flow_to_draw)}")

DEBUG: Počet riadkov v country_hubs: 225
DEBUG: Prvé 3 indexy v country_hubs: ['Afghanistan', 'Albania', 'Algeria']
POČET MEDZINÁRODNÝCH TOKOV NA VYKRESLENIE: 4558


In [68]:
mapa_v2 = Map(tiles="cartodb positron") # Na bielom podklade hneda vynikne
mapa_v2.add_nodes(country_hubs)
mapa_v2.add_edges(flow_to_draw, scaling=2.5,offset_dist=0.5)
mapa_v2.showMap("letecke_toky_hnedo.html")

In [69]:
# 1. Odfiltrujeme len interkontinentálne lety
continental_edges = edges[edges["src_continent"] == edges["dest_continent"]].copy()

# 2. Spočítame traffic pre každé letisko, ale len z interkontinentálnych letov
# Spočítame výskyty v zdroji aj cieli
src_traffic = continental_edges.groupby("# source").size()
dest_traffic = continental_edges.groupby("target").size()
in_traffic = src_traffic.add(dest_traffic, fill_value=0)

# 3. Pridáme tento špecifický traffic do nodes
nodes["in_traffic"] = nodes.index.map(in_traffic).fillna(0)

# 4. Pre každý kontinent nájdeme letisko s NAJVÄČŠÍM in_traffic
continent_hubs = (
    nodes.sort_values(["continent", "in_traffic"], ascending=[True, False])
    .groupby("continent")
    .head(5)
    .copy()
)

continent_hubs["continent"] = continent_hubs["continent"].astype(str).str.strip()
continent_hubs.set_index("continent", inplace=True)
continent_hubs = continent_hubs[["name", "city", "latitude", "longitude", "in_traffic"]].rename(
    columns={"name": "hub_name"}
)

In [70]:
continental_place_mapping = {
    # Africa
    "OR Tambo International Airport": "bottom",
    "Jomo Kenyatta International Airport": "right",
    "Addis Ababa Bole International Airport": "up",
    "Mohammed V International Airport": "bottom",
    "Port Bouet Airport": "bottom",
    
    # Asia (veľmi husté v Číne)
    "Beijing Capital International Airport": "up",
    "Shanghai Pudong International Airport": "right",
    "Singapore Changi Airport": "bottom",
    "Chengdu Shuangliu International Airport": "left",
    "Guangzhou Baiyun International Airport": "bottom",
    
    # Europe (husté stredné a južné krídlo)
    "Barcelona International Airport": "left",
    "Palma De Mallorca Airport": "right",
    "Amsterdam Airport Schiphol": "up",
    "Frankfurt am Main Airport": "left",
    "Munich Airport": "right",
    
    # North America
    "Hartsfield Jackson Atlanta International Airport": "right",
    "Chicago O'Hare International Airport": "right",
    "Dallas Fort Worth International Airport": "bottom",
    "Los Angeles International Airport": "left",
    "Denver International Airport": "up",
    
    # Oceania
    "Sydney Kingsford Smith International Airport": "right",
    "Brisbane International Airport": "up",
    "Melbourne International Airport": "left",
    "Auckland International Airport": "bottom",
    "Perth International Airport": "left",
    
    # South America
    "Guarulhos - Governador André Franco Montoro International Airport": "left",
    "El Dorado International Airport": "up",
    "Presidente Juscelino Kubistschek International Airport": "up",
    "Jorge Chávez International Airport": "left",
    "Rio Galeão – Tom Jobim International Airport": "right"
}
continental_anchor_mapping = {
    "up": (79, 75),      # Špička dole v strede
    "bottom": (79, 0),   # Špička hore v strede
    "left": (160, 30),   # Špička vpravo v strede
    "right": (0, 32)     # Špička vľavo v strede
}
# 1. Priradíme smer
continent_hubs["place"] = continent_hubs["hub_name"].map(continental_place_mapping).fillna("up")

# 2. Priradíme kotvu na základe smeru
continent_hubs["anchor"] = continent_hubs["place"].map(continental_anchor_mapping)

In [ ]:
# 1. Agregácia tokov podľa kontinentov (použijeme pôvodné edges s kontinentmi)
flow_df_continent = (
    edges.groupby(["src_continent", "dest_continent"])
    .size()
    .reset_index(name="flight_count")
)
# Definujeme vizuálne stredy kontinentov (X, Y)
visual_centers = {
    "Europe": [61.0, 15.0],
    "Asia": [55.0, 95.0],
    "Africa": [-6.5, 26.0],
    "North America": [55.0, -100.0],
    "South America": [-35.0, -65.0],
    "Oceania": [-20.0, 136.0],
}


def get_visual_center(cont_name, shift=0):
    cont_name = str(cont_name).strip()
    if cont_name in visual_centers:
        coords = visual_centers[cont_name]
        return f"{coords[0] },{coords[1]+shift}"
    return None


# Aplikujeme tieto stredy na flow dataframe
flow_df_continent[["latitude", "longitude"]] = (
    flow_df_continent["src_continent"]
    .apply(get_visual_center, shift=-3)
    .str.split(",", expand=True)
)

flow_df_continent = flow_df_continent[flow_df_continent["dest_continent"] == flow_df_continent["src_continent"]]
intra_continent_df = flow_df_continent.dropna(subset=["longitude", "latitude"]).rename(
    columns={"src_continent": "city", "flight_count": "in_traffic"})
continent_summary_place_mapping = {
    "Africa": "bottom",
    "Asia": "up",
    "Europe": "up",
    "North America": "up",
    "Oceania": "left",
    "South America": "bottom"
}
continent_summary_anchor_mapping = {
    "up": (79, 75),
    "bottom": (79, 0),
    "left": (160, 30),
    "right": (0, 32)
}
# 1. Priradíme smer (podľa stĺpca 'city', kde máš názov kontinentu)
intra_continent_df["place"] = intra_continent_df["city"].map(continent_summary_place_mapping).fillna("up")

# 2. Priradíme kotvu
intra_continent_df["anchor"] = intra_continent_df["place"].map(continent_summary_anchor_mapping)

In [72]:
mapa_v3 = Map(zoom_start=3,tiles="cartodb positron") # Na bielom podklade hneda vynikne
mapa_v3.add_nodes(pd.concat([continent_hubs,intra_continent_df]),True)
mapa_v3.showMap("letecke_toky_kontinenty.html")

In [73]:
# 1. Odfiltrujeme len interkontinentálne lety
intercontinental_edges = edges[edges["src_continent"] != edges["dest_continent"]].copy()

# 2. Spočítame traffic pre každé letisko, ale len z interkontinentálnych letov
# Spočítame výskyty v zdroji aj cieli
src_traffic = intercontinental_edges.groupby("# source").size()
dest_traffic = intercontinental_edges.groupby("target").size()
gateway_traffic = src_traffic.add(dest_traffic, fill_value=0)

# 3. Pridáme tento špecifický traffic do nodes
nodes["gateway_traffic"] = nodes.index.map(gateway_traffic).fillna(0)

# 4. Pre každý kontinent nájdeme letisko s NAJVÄČŠÍM gateway_traffic
gateway_hubs = (
    nodes.sort_values(["continent", "gateway_traffic"], ascending=[True, False])
    .groupby("continent")
    .head(3)
    .copy()
)

# 5. Vyčistíme a nastavíme index na continent
gateway_hubs["continent"] = gateway_hubs["continent"].astype(str).str.strip()
gateway_hubs.set_index("continent", inplace=True)
gateway_hubs = gateway_hubs[["name", "city", "latitude", "longitude", "gateway_traffic"]].rename(
    columns={"name": "hub_name"}
)

print("Nájdené interkontinentálne brány:")
display(gateway_hubs)

Nájdené interkontinentálne brány:


,hub_name,city,latitude,longitude,gateway_traffic
continent,,,,,
Africa,Cairo International Airport,Cairo,30.121901,31.405600,213.0
Africa,Mohammed V International Airport,Casablanca,33.367500,-7.589970,154.0
Africa,Menara Airport,Marrakech,31.606899,-8.036300,142.0
Asia,Atatürk International Airport,Istanbul,40.976898,28.814600,386.0
Asia,Dubai International Airport,Dubai,25.252800,55.364399,315.0
Asia,Narita International Airport,Tokyo,35.764702,140.386002,233.0
Europe,London Heathrow Airport,London,51.470600,-0.461941,640.0
Europe,Charles de Gaulle International Airport,Paris,49.012798,2.550000,557.0
Europe,Frankfurt am Main Airport,Frankfurt,50.033333,8.570556,468.0


In [74]:
# 1. Agregácia tokov podľa kontinentov (použijeme pôvodné edges s kontinentmi)
flow_df_continent = (
    edges.groupby(["src_continent", "dest_continent"])
    .size()
    .reset_index(name="flight_count")
)
visual_centers = {
    'Europe': [50.0, 15.0],
    'Asia': [45.0, 105.0],
    'Africa': [-15.0, 25.0],
    'North America': [55.0, -100.0],
    'South America': [-15.0, -60.0],
    'Oceania': [-25.0, 135.0]
}

def get_visual_center(cont_name):
    cont_name = str(cont_name).strip()
    if cont_name in visual_centers:
        coords = visual_centers[cont_name]
        return f"{coords[0]},{coords[1]}"
    return None
flow_to_draw_continent = flow_df_continent[
    flow_df_continent["src_continent"] != flow_df_continent["dest_continent"]
]
# Aplikujeme tieto stredy na flow dataframe
flow_to_draw_continent["start"] = flow_to_draw_continent["src_continent"].apply(get_visual_center)
flow_to_draw_continent["end"] = flow_to_draw_continent["dest_continent"].apply(get_visual_center)

# Vyčistíme prípadné riadky, ktoré by nemali priradený stred
flow_to_draw_continent = flow_to_draw_continent.dropna(subset=["start", "end"])

In [75]:
place_mapping = {
    # Europe
    "London Heathrow Airport": ("up", (79, 80)),
    "Charles de Gaulle International Airport": ("left", (160, 32)),
    "Frankfurt am Main Airport": ("right", (0, 32)),
    
    # North America
    "John F Kennedy International Airport": ("up", (79, 75)),
    "Miami International Airport": ("right", (0, 32)),
    "Los Angeles International Airport": ("right", (0, 32)),
    
    # Africa
    "Cairo International Airport": ("right", (0, 32)),
    "Mohammed V International Airport": ("up", (79, 75)),
    "Menara Airport": ("left", (160, 32)),
    
    # Asia
    "Atatürk International Airport": ("right", (0, 32)),
    "Dubai International Airport": ("bottom", (79, 0)),
    "Narita International Airport": ("bottom", (79, 0)),
    
    # South America
    "El Dorado International Airport": ("left", (160,32)),
    "Guarulhos - Governador André Franco Montoro International Airport": ("bottom", (79, 0)),
    "Jorge Chávez International Airport": ("bottom", (79, 0)),
    
    # Oceania
    "Sydney Kingsford Smith International Airport": ("right", (0, 32)),
    "Melbourne International Airport": ("left", (160, 32)),
    "Perth International Airport": ("left", (160, 32))
}
# Namapujeme celý záznam (tuple)
gateway_hubs["mapped_data"] = gateway_hubs["hub_name"].map(place_mapping)

# Rozdelíme ho na place a anchor (všetko bez funkcií, len prístup k prvkom)
gateway_hubs["place"] = gateway_hubs["mapped_data"].str[0]
gateway_hubs["anchor"] = gateway_hubs["mapped_data"].str[1]

# Voliteľné: zmažeme dočasný stĺpec
gateway_hubs = gateway_hubs.drop(columns=["mapped_data"])

In [76]:
# Inicializácia
mapa_gateways = Map(tiles="cartodb positron", zoom_start=3)

# Pridáme uzly (Brány)
# Môžeme ich vykresliť výraznejšie
mapa_gateways.add_nodes(gateway_hubs, True) # Metóda funguje aj pre kontinenty, ak sedí formát DF

# Pridáme toky
mapa_gateways.add_edges(flow_to_draw_continent, scaling=4.0, offset_dist=3, value_show=True)

# Uložiť a zobraziť
mapa_gateways.showMap("intercontinental_gateways.html")

In [77]:
# Tvoj zoznam indexov
route_indices = [2317, 2287, 2330, 2292, 57, 86, 125, 770, 461, 464, 468, 3205, 466, 465]

# Vytvoríme DataFrame pre lety (Source -> Target)
route_flights = []
for i in range(len(route_indices) - 1):
    src_idx = route_indices[i]
    dest_idx = route_indices[i+1]
    
    # Získame súradnice z nodes
    src_row = nodes.loc[src_idx]
    dest_row = nodes.loc[dest_idx]
    
    route_flights.append({
        "start": f"{src_row['latitude']},{src_row['longitude']}",
        "end": f"{dest_row['latitude']},{dest_row['longitude']}",
        "flight_count": 100, # Nastavíme fixnú váhu, aby bola čiara pekne viditeľná
        "src_name": src_row.get('name', src_idx),
        "dest_name": dest_row.get('name', dest_idx)
    })

df_route = pd.DataFrame(route_flights)

In [78]:
df_route

,start,end,flight_count,src_name,dest_name
0,"54.98809814453125,-85.44329833984375","52.9275016784668,-82.43190002441406",100,Peawanuck Airport,Attawapiskat Airport
1,"52.9275016784668,-82.43190002441406","52.282501220703125,-81.67780303955078",100,Attawapiskat Airport,Kashechewan Airport
2,"52.282501220703125,-81.67780303955078","52.20140075683594,-81.6968994140625",100,Kashechewan Airport,Fort Albany Airport
3,"52.20140075683594,-81.6968994140625","51.29109954833984,-80.60780334472656",100,Fort Albany Airport,Moosonee Airport
4,"51.29109954833984,-80.60780334472656","48.5696983337,-81.376701355",100,Moosonee Airport,Timmins/Victor M. Power
5,"48.5696983337,-81.376701355","43.6772003174,-79.63059997559999",100,Timmins/Victor M. Power,Lester B. Pearson International Airport
6,"43.6772003174,-79.63059997559999","40.976898,28.8146",100,Lester B. Pearson International Airport,Atatürk International Airport
7,"40.976898,28.8146","-4.38575,15.4446",100,Atatürk International Airport,Ndjili International Airport
8,"-4.38575,15.4446","0.481638997793,25.3379993439",100,Ndjili International Airport,Bangoka International Airport
9,"0.481638997793,25.3379993439","-1.670809984207153,29.238500595092773",100,Bangoka International Airport,Goma International Airport
